# Create Shandong NSF Awards (Natural Science Foundation of Shandong Province)

Creates awards from the Shandong provincial S&T department's public
自然科学基金 拟立项/拟推荐 rosters (立项公示) harvested off kjt.shandong.gov.cn.

**Prerequisites:**
- Run `scripts/local/shandong_nsf_to_s3.py` first (thin runner over the shared
  `scripts/local/cn_provincial` framework). It uploads
  `s3://openalex-ingest/awards/shandong_nsf/shandong_nsf_projects.parquet`.

**Data source:** http://kjt.shandong.gov.cn/col/col13360/index.html (通知公告 column).
Public roster window is **2014-2020** (PDF/xls/xlsx/docx attachments listing
项目名称 / 申报者 / 依托单位 [/ 项目类别]). From 2021 the rosters moved behind the
cloud.kjt.shandong.gov.cn login, so 2021+ batches are not publicly harvestable.

**Amounts:** the rosters publish **no funding amounts** -> `amount`/`currency`
are NULL. **§6.7 amount-coverage check is waived** for this funder: provincial
NSF grants carry implicit standard tiers and the announcements only list
title/PI/institution. This is the documented waiver, not a mapping miss.

**PI names:** Chinese, family-first. Per the NSFC precedent the full name is
stored in `family_name` with `given_name` NULL (Chinese order is unsplittable
without a dictionary).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320324174`
- display_name: Natural Science Foundation of Shandong Province
- ror_id / doi: from `openalex.common.funder`
- country: CN

**Priority:** `439` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).


## Step 1: Create Staging Table from S3


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.shandong_nsf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/shandong_nsf/shandong_nsf_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.shandong_nsf_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.shandong_nsf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.shandong_nsf_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320324174 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.common.funder`. If 0 rows, STOP (do not proceed) and flag.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320324174;

## Step 2: Create Shandong NSF Awards Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.shandong_nsf_awards
USING delta
AS
WITH
-- Path A: Shandong NSF is F4320* (Crossref-registered) -> resolve from the dim.
sd_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320324174  -- Natural Science Foundation of Shandong Province
),

awards_transformed AS (
    SELECT
        -- Unique id. funder_award_id (申报编号) is present on only a minority of
        -- rosters, so key the hash on funder_award_id when we have it, else on a
        -- synthetic (title + institution) key. This keeps ids stable across re-ingests.
        abs(xxhash64(CONCAT(
            f.funder_id, ':',
            COALESCE(
                NULLIF(LOWER(TRIM(g.funder_award_id)), ''),
                CONCAT(LOWER(TRIM(g.display_name)), '|', LOWER(TRIM(COALESCE(g.institution, ''))))
            )
        ))) % 9000000000 as id,

        g.display_name as display_name,
        CAST(NULL AS STRING) as description,

        f.funder_id,
        NULLIF(TRIM(g.funder_award_id), '') as funder_award_id,

        -- Amount: rosters publish none -> NULL (§6.7 waiver, documented in header).
        CAST(NULL AS DOUBLE) as amount,
        CAST(NULL AS STRING) as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        -- Funding type from the batch/scheme label (青年/优青/杰青 -> fellowship;
        -- 重点/重大/联合基金 -> research; else the default 'grant').
        CASE
            WHEN g.funder_scheme LIKE '%杰出青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%优秀青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%博士基金%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%重大%' THEN 'research'
            WHEN g.funder_scheme LIKE '%重点%' THEN 'research'
            WHEN g.funder_scheme LIKE '%联合基金%' THEN 'research'
            ELSE 'grant'
        END as funding_type,

        NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,

        'shandong_nsf' as provenance,

        -- Dates: only the approval year is published. start = yyyy-01-01, no end.
        CASE WHEN TRY_CAST(g.start_year AS INT) IS NOT NULL
             THEN TRY_TO_DATE(CONCAT(g.start_year, '-01-01'), 'yyyy-MM-dd')
             ELSE NULL END as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.start_year AS INT) as start_year,
        CAST(NULL AS INT) as end_year,

        -- Lead investigator: Chinese PI full name in family_name, given NULL (NSFC precedent).
        -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
        CASE
            WHEN (g.lead_family_name IS NOT NULL AND TRIM(g.lead_family_name) != '')
              OR (g.institution IS NOT NULL AND TRIM(g.institution) != '') THEN
                struct(
                    CAST(NULL AS STRING) as given_name,
                    NULLIF(TRIM(g.lead_family_name), '') as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        NULLIF(TRIM(g.institution), '') as name,
                        'China' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        -- Landing page = the announcement article the roster came from.
        g.landing_page_url as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(
                   f.funder_id, ':',
                   COALESCE(
                       NULLIF(LOWER(TRIM(g.funder_award_id)), ''),
                       CONCAT(LOWER(TRIM(g.display_name)), '|', LOWER(TRIM(COALESCE(g.institution, ''))))
                   )
               ))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM openalex.awards.shandong_nsf_raw g
    CROSS JOIN sd_funder f
    WHERE g.display_name IS NOT NULL
      AND TRIM(g.display_name) != ''
)

SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'shandong_nsf' AND priority = 439;

-- Insert into openalex_awards_raw with priority.
-- Priority 439: direct-from-funder ingest of Shandong NSF's own public rosters.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 439 outranks the
-- acknowledgement shells (priority 0) and grant-DOI stubs (priority 1).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    439 as priority
FROM openalex.awards.shandong_nsf_awards;

## Verification Queries


In [ ]:
%sql
SELECT COUNT(*) as total_shandong_nsf_awards FROM openalex.awards.shandong_nsf_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency,
       start_year, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.shandong_nsf_awards LIMIT 20;

In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt FROM openalex.awards.shandong_nsf_awards
GROUP BY funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.shandong_nsf_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- §6.7 coverage. amount is intentionally 0% (rosters publish no amounts -> waiver).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.shandong_nsf_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (§6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'shandong_nsf'
GROUP BY provenance, priority;